In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126    

In [ ]:
import torch
from transformers import LlamaTokenizer, LlamaForCausalLM
from huggingface_hub import login
import sys
import gc
import transformers  # For version check

# Print environment info for debugging
print(f"Python version: {sys.version}")
print(f"Torch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

# Hugging Face login
def huggingface_login():
    print("Enter your Hugging Face token (from https://huggingface.co/settings/tokens):")
    token = input("Token: ").strip()
    try:
        login(token)
        print("Logged into Hugging Face successfully!")
    except Exception as e:
        print(f"Login failed: {e}")
        sys.exit(1)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Detected device: {device}")

# Model and tokenizer setup
model_name = "meta-llama/Llama-2-7b-hf"
try:
    print("Step 1: Loading tokenizer...")
    tokenizer = LlamaTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        print("Padding token set to EOS token.")
    print("Tokenizer loaded successfully.")

    print("Step 2: Loading model...")
    model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,  # FP16 for efficiency
        low_cpu_mem_usage=True,
        device_map="auto"           # Auto-distribute
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Loading failed: {e}")
    sys.exit(1)

def generate_response(prompt, max_new_tokens=50):
    """Generate a response with step-by-step debugging."""
    print(f"\nGenerating response for prompt: '{prompt}'")
    try:
        # Step 1: Tokenize input
        print("Tokenizing input...")
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        print(f"Input IDs: {input_ids}")
        print(f"Input IDs shape: {input_ids.shape}")
        print(f"Attention mask: {attention_mask}")

        # Step 2: Generate output
        print("Generating output...")
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
        print(f"Generated output IDs: {outputs}")
        print(f"Output shape: {outputs.shape}")

        # Step 3: Decode output
        print("Decoding output...")
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Raw decoded response: '{response}'")

        # Clean up
        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()
        return response.strip()
    except Exception as e:
        print(f"Generation failed: {e}")
        return "Error generating response."

# Test the code
if __name__ == "__main__":
  while True:
    # Get user input
        user_input = input("\nYou: ").strip()

        # Check for exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Exiting chat...")
            break

        # Generate and display response
        print(f"Processing input: '{user_input}'")
        response = generate_response(user_input)
        print(f"LLaMA 2: '{response}'")

 # Memory cleanup
print("Memory cleanup...")
torch.cuda.empty_cache()
gc.collect()

print("Chat ended. Final memory cleanup...")
torch.cuda.empty_cache()
gc.collect()
print("Done.")